# 📦 Notebook 1: Chunked File Uploads

Dropbox lets you upload files of **any size** — even 50 GB videos. But you can't just send 50 GB in one HTTP request. In this notebook we'll learn how real systems break large files into small **chunks**, upload them independently, and reassemble them on the server.

Think of it like mailing a book one chapter at a time: if one envelope gets lost, you only resend that chapter — not the whole book.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why large files can't be uploaded in a single request
- How to split a file into fixed-size chunks and fingerprint each one
- What presigned URLs are and why they let clients upload directly to object storage
- How to track chunk upload status in a database
- How resumable uploads work (only re-upload what failed)
- How to reassemble chunks and verify data integrity

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/dropbox
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `dropbox_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`
- **MinIO Console** (S3-compatible storage GUI): http://localhost:9001  
  Login: User `minioadmin`, Password `minioadmin`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import boto3
import hashlib
import os
import io
import time
import requests

# ── PostgreSQL connection settings ──
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "dropbox_demo",
    "user": "demo",
    "password": "demo"
}

# ── Redis connection settings ──
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

# ── MinIO (S3-compatible) connection settings ──
S3_CONFIG = {
    "endpoint_url": "http://localhost:9000",
    "aws_access_key_id": "minioadmin",
    "aws_secret_access_key": "minioadmin",
}
BUCKET_NAME = "dropbox-files"

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

def get_s3_client():
    return boto3.client("s3", **S3_CONFIG)

# ── Test all three connections ──
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

try:
    s3 = get_s3_client()
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"✅ Connected to MinIO (bucket: {BUCKET_NAME})")
except Exception as e:
    print(f"❌ MinIO failed: {e}")
    print("   Run: docker compose up -d")

## 🤔 Why Can't We Upload a Whole File at Once?

Imagine you're uploading a 50 GB video to Dropbox. Here's what would happen with a single HTTP POST:

| Problem | Why It Hurts |
|---------|-------------|
| **Timeouts** | Most API gateways (Nginx, AWS ALB) cap requests at 10–60 seconds |
| **Payload limits** | Many load balancers reject bodies larger than 10 MB |
| **Network hiccups** | If your WiFi drops at 49 GB, you start over from scratch |
| **Memory pressure** | The server would need to buffer 50 GB in RAM |

Let's do some quick math to see why this matters.

In [ ]:
# Quick math: how long does it take to upload a 50 GB file?

file_size_gb = 50
file_size_bytes = file_size_gb * 1024 ** 3  # convert GB → bytes (binary GB)
file_size_bits = file_size_bytes * 8          # convert bytes → bits

speeds = [
    ("Home WiFi (50 Mbps)",    50_000_000),
    ("Fast broadband (100 Mbps)", 100_000_000),
    ("Gigabit fiber (1 Gbps)",  1_000_000_000),
]

print(f"📐 Uploading a {file_size_gb} GB file ({file_size_bytes:,} bytes)")
print("=" * 55)

minutes_by_label = {}
for label, speed_bps in speeds:
    seconds = file_size_bits / speed_bps
    minutes = seconds / 60
    minutes_by_label[label] = minutes
    print(f"  {label:<30} → {minutes:>6.1f} min ({seconds:,.0f} sec)")

broadband_min = minutes_by_label["Fast broadband (100 Mbps)"]

print()
print("⚠️  A typical API gateway timeout is 30–60 seconds.")
print(f"    Even on fast broadband, 50 GB takes ~{broadband_min:.0f} minutes!")
print()
print("💡 Solution: break the file into small chunks (e.g., 1 MB each)")
print(f"   50 GB ÷ 1 MB = {file_size_bytes // (1024**2):,} chunks")
print("   Each chunk uploads in under 1 second — well within timeout limits.")

## ✂️ Splitting a File into Chunks

The first step is to split our file into fixed-size pieces. Each piece is called a **chunk**.

For each chunk we also compute a **fingerprint** — a SHA-256 hash that uniquely identifies the content. This lets us:
- Verify nothing was corrupted during upload
- Detect duplicate chunks (deduplication)
- Confirm the reassembled file matches the original

Let's create a sample file and split it.

In [ ]:
# ── Step 1: Create a sample file (5 MB of random-looking text) ──

SAMPLE_SIZE = 5 * 1024 * 1024  # 5 MB

# We use a repeating pattern so the file is reproducible,
# but large enough to demonstrate chunking.
line = "The quick brown fox jumps over the lazy dog. " * 5 + "\n"
file_content = (line * (SAMPLE_SIZE // len(line) + 1))[:SAMPLE_SIZE]
file_bytes = file_content.encode("utf-8")

print(f"📄 Created sample file")
print(f"   Size: {len(file_bytes):,} bytes ({len(file_bytes) / 1024 / 1024:.2f} MB)")
print(f"   First 80 chars: {file_content[:80]}...")

In [ ]:
# ── Step 2: Split into 1 MB chunks and compute fingerprints ──

CHUNK_SIZE = 1 * 1024 * 1024  # 1 MB per chunk

def split_into_chunks(data: bytes, chunk_size: int) -> list:
    """Split raw bytes into fixed-size chunks.
    Returns a list of (chunk_index, chunk_bytes, sha256_hex) tuples.
    """
    chunks = []
    for i in range(0, len(data), chunk_size):
        chunk = data[i : i + chunk_size]
        fingerprint = hashlib.sha256(chunk).hexdigest()
        chunks.append((len(chunks), chunk, fingerprint))
    return chunks

def file_fingerprint(data: bytes) -> str:
    """Compute a SHA-256 fingerprint for the entire file."""
    return hashlib.sha256(data).hexdigest()

# Split our sample file
chunks = split_into_chunks(file_bytes, CHUNK_SIZE)
whole_file_fp = file_fingerprint(file_bytes)

print(f"✂️  Split into {len(chunks)} chunks (each ≤ {CHUNK_SIZE // 1024} KB)")
print(f"📋 Whole-file fingerprint: {whole_file_fp[:16]}...")
print()
print(f"{'Chunk':<7} {'Size':>10} {'Fingerprint (first 16 chars)'}")
print("-" * 50)
for idx, chunk_data, fp in chunks:
    print(f"  {idx:<5} {len(chunk_data):>10,} B   {fp[:16]}...")

assert b"".join(c for _, c, _ in chunks) == file_bytes, "chunking must not lose or reorder bytes"
assert [i for i, _, _ in chunks] == list(range(len(chunks))), "chunk indices must be dense and ordered"

## 🔑 Presigned URLs — Let Clients Upload Directly to Storage

In a real system, the client (your browser or desktop app) doesn't send file data **through** the backend server. That would make the backend a bottleneck.

Instead the flow looks like this:

```
Client                    Backend (API)              MinIO / S3
  │                           │                          │
  │  1. "I want to upload     │                          │
  │      report.pdf (5 MB)"   │                          │
  │ ─────────────────────────>│                          │
  │                           │  2. Generate presigned   │
  │                           │     PUT URLs for each    │
  │                           │     chunk                │
  │  3. Here are your URLs    │                          │
  │ <─────────────────────────│                          │
  │                           │                          │
  │  4. PUT chunk data ──────────────────────────────────>│
  │     (directly to S3!)     │                          │
  │ <────────────────────────────────────────────────────│
  │  5. 200 OK + ETag         │                          │
```

A **presigned URL** is a temporary URL that includes a cryptographic signature. It lets anyone with the URL upload (or download) a specific object — no AWS credentials needed. The URL expires after a set time (e.g., 1 hour).

Let's generate one and use it.

In [ ]:
# ── Generate a presigned PUT URL for one chunk ──

s3 = get_s3_client()

# The storage key is where the chunk will live in the bucket.
# Convention: uploads/<file_id>/chunk_<index>
demo_key = "uploads/demo/chunk_0"

presigned_url = s3.generate_presigned_url(
    ClientMethod="put_object",
    Params={"Bucket": BUCKET_NAME, "Key": demo_key},
    ExpiresIn=3600,  # URL valid for 1 hour
)

print("🔑 Presigned PUT URL (valid for 1 hour):")
print(f"   {presigned_url[:80]}...")
print()
print("   Notice the URL contains:")
print("   • The bucket and key path")
print("   • X-Amz-Signature — the cryptographic proof")
print("   • X-Amz-Expires — when the URL stops working")

In [ ]:
# ── Upload one chunk using the presigned URL ──
# In a real app, the *client* (browser/mobile) does this HTTP PUT directly.
# The backend never touches the file bytes!

chunk_index, chunk_data, chunk_fp = chunks[0]  # first chunk

response = requests.put(presigned_url, data=chunk_data)

print(f"📤 Uploaded chunk 0 via presigned URL")
print(f"   HTTP status: {response.status_code}")
print(f"   ETag:        {response.headers.get('ETag', 'N/A')}")
print(f"   Chunk size:  {len(chunk_data):,} bytes")
print()
if response.status_code == 200:
    print("✅ Success! The chunk is now stored in MinIO.")
    print("   Open http://localhost:9001 to see it in the MinIO Console.")
else:
    print(f"❌ Upload failed: {response.text}")

# Clean up the demo object
s3.delete_object(Bucket=BUCKET_NAME, Key=demo_key)

## 🗄️ Tracking Chunk Status in Postgres

When the client starts an upload, the backend does three things:

1. **Creates a `files` row** with `status = 'uploading'`
2. **Creates one `chunks` row per chunk** with `status = 'pending'`
3. **Returns presigned URLs** for every chunk

As each chunk finishes uploading, the client tells the backend, and we flip that chunk's status to `'uploaded'`. When **all** chunks are uploaded, we mark the file as `'uploaded'`.

Let's walk through this entire flow step by step.

In [ ]:
# ── Step 1: Initiate the upload (backend creates metadata) ──

def initiate_upload(owner_id: int, file_name: str, file_size: int,
                    fingerprint: str, chunk_infos: list) -> dict:
    """
    Register a new file upload in Postgres and return presigned URLs.
    
    chunk_infos: list of (chunk_index, chunk_size, fingerprint) tuples
    """
    conn = get_db_connection()
    conn.autocommit = False
    cur = conn.cursor()
    s3 = get_s3_client()
    
    try:
        # Insert the file record
        cur.execute("""
            INSERT INTO files (owner_id, file_name, file_size, fingerprint, status)
            VALUES (%s, %s, %s, %s, 'uploading')
            RETURNING id
        """, (owner_id, file_name, file_size, fingerprint))
        file_id = cur.fetchone()[0]
        
        # Insert a row for each chunk and generate presigned URLs
        presigned_urls = []
        for idx, csize, cfp in chunk_infos:
            storage_key = f"uploads/{file_id}/chunk_{idx}"
            cur.execute("""
                INSERT INTO chunks (file_id, chunk_index, chunk_size, fingerprint, storage_key, status)
                VALUES (%s, %s, %s, %s, %s, 'pending')
            """, (file_id, idx, csize, cfp, storage_key))
            
            url = s3.generate_presigned_url(
                ClientMethod="put_object",
                Params={"Bucket": BUCKET_NAME, "Key": storage_key},
                ExpiresIn=3600,
            )
            presigned_urls.append({"chunk_index": idx, "url": url, "storage_key": storage_key})
        
        conn.commit()
        print(f"✅ Upload initiated  — file_id={file_id}, {len(chunk_infos)} chunks registered")
        return {"file_id": file_id, "presigned_urls": presigned_urls}
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

# Prepare chunk metadata
chunk_infos = [(idx, len(data), fp) for idx, data, fp in chunks]

# Alice (user_id=1) initiates an upload
upload = initiate_upload(
    owner_id=1,
    file_name="big_report.txt",
    file_size=len(file_bytes),
    fingerprint=whole_file_fp,
    chunk_infos=chunk_infos,
)

FILE_ID = upload["file_id"]
print(f"   File ID:    {FILE_ID}")
print(f"   Chunks:     {len(upload['presigned_urls'])}")
print(f"   First URL:  {upload['presigned_urls'][0]['url'][:60]}...")

In [ ]:
# Let's peek at what Postgres looks like right now

conn = get_db_connection()
cur = conn.cursor()

print("📋 files table:")
cur.execute("SELECT id, file_name, file_size, status FROM files WHERE id = %s", (FILE_ID,))
row = cur.fetchone()
print(f"   id={row[0]}, name={row[1]}, size={row[2]:,}, status={row[3]}")
print()

print("📋 chunks table:")
cur.execute("""
    SELECT chunk_index, chunk_size, status, storage_key
    FROM chunks WHERE file_id = %s ORDER BY chunk_index
""", (FILE_ID,))
print(f"   {'Idx':<5} {'Size':>10} {'Status':<12} {'Storage Key'}")
print(f"   {'-'*55}")
for r in cur.fetchall():
    print(f"   {r[0]:<5} {r[1]:>10,} {r[2]:<12} {r[3]}")

conn.close()
print()
print("👆 All chunks are 'pending' — nothing has been uploaded to MinIO yet.")

In [ ]:
# ── Step 2: Upload each chunk via its presigned URL ──

def upload_chunk(presigned_url: str, chunk_data: bytes) -> str:
    """Upload chunk bytes to MinIO using the presigned PUT URL.
    Returns the ETag on success, or raises on failure.
    """
    resp = requests.put(presigned_url, data=chunk_data)
    resp.raise_for_status()
    return resp.headers.get("ETag", "")

def mark_chunk_uploaded(file_id: int, chunk_index: int, etag: str):
    """Update a chunk's status to 'uploaded' after successful upload."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("""
        UPDATE chunks
        SET status = 'uploaded', etag = %s, uploaded_at = NOW()
        WHERE file_id = %s AND chunk_index = %s
    """, (etag, file_id, chunk_index))
    conn.commit()
    conn.close()

# Upload all chunks
print("📤 Uploading chunks...")
for info in upload["presigned_urls"]:
    idx = info["chunk_index"]
    _, chunk_data, _ = chunks[idx]
    
    etag = upload_chunk(info["url"], chunk_data)
    mark_chunk_uploaded(FILE_ID, idx, etag)
    print(f"   ✅ Chunk {idx} uploaded ({len(chunk_data):,} bytes)")

print()
print("🎉 All chunks uploaded!")

In [ ]:
# ── Step 3: Finalize — mark the file as 'uploaded' ──

def finalize_upload(file_id: int) -> bool:
    """Check if all chunks are uploaded and mark the file as done."""
    conn = get_db_connection()
    cur = conn.cursor()
    
    # Count total vs uploaded chunks
    cur.execute("SELECT COUNT(*) FROM chunks WHERE file_id = %s", (file_id,))
    total = cur.fetchone()[0]
    
    cur.execute("SELECT COUNT(*) FROM chunks WHERE file_id = %s AND status = 'uploaded'", (file_id,))
    uploaded = cur.fetchone()[0]
    
    if uploaded == total:
        cur.execute("""
            UPDATE files SET status = 'uploaded', updated_at = NOW()
            WHERE id = %s
        """, (file_id,))
        conn.commit()
        conn.close()
        print(f"✅ File {file_id} finalized: {uploaded}/{total} chunks uploaded")
        return True
    else:
        conn.close()
        print(f"⏳ File {file_id} incomplete: {uploaded}/{total} chunks uploaded")
        return False

finalize_upload(FILE_ID)

In [ ]:
# Let's verify the final state in Postgres

conn = get_db_connection()
cur = conn.cursor()

cur.execute("SELECT id, file_name, file_size, status FROM files WHERE id = %s", (FILE_ID,))
row = cur.fetchone()
print(f"📋 File: id={row[0]}, name={row[1]}, size={row[2]:,}, status={row[3]}")
print()

cur.execute("""
    SELECT chunk_index, status, uploaded_at
    FROM chunks WHERE file_id = %s ORDER BY chunk_index
""", (FILE_ID,))
print(f"   {'Idx':<5} {'Status':<12} {'Uploaded At'}")
print(f"   {'-'*45}")
for r in cur.fetchall():
    print(f"   {r[0]:<5} {r[1]:<12} {r[2]}")

cur.execute("""
    SELECT COUNT(*) FROM chunks WHERE file_id = %s AND status <> 'uploaded'
""", (FILE_ID,))
not_done = cur.fetchone()[0]
conn.close()
print()
print("🎉 Everything is 'uploaded' — the file and all its chunks are complete!")

assert row[3] == "uploaded", f"file should be finalized, status is {row[3]!r}"
assert not_done == 0, f"{not_done} chunk(s) never reached 'uploaded'"

## 🔄 Resumable Uploads — Don't Start Over!

What happens if your internet drops mid-upload? Or your laptop closes? With chunked uploads, you don't lose everything.

The idea is simple:
1. Ask the server: *"which chunks are still pending?"*
2. Only upload those chunks
3. Skip the ones that already succeeded

This is like checking off chapters you've already mailed. If chapters 1–3 arrived but chapter 4 got lost, you only resend chapter 4.

Let's break an upload **for real** — we'll hand the last two chunks a presigned URL with a
corrupted signature, so MinIO rejects the PUT and those chunks stay `pending` — and then resume it.

In [ ]:
# ── Simulate a new upload that partially fails ──

# Start a fresh upload for the same file (as if Alice uploads another file)
upload2 = initiate_upload(
    owner_id=1,
    file_name="vacation_video.mp4",
    file_size=len(file_bytes),
    fingerprint=whole_file_fp,
    chunk_infos=chunk_infos,
)
FILE_ID_2 = upload2["file_id"]

# Chunks 0-2 go up normally. Chunks 3-4 hit a *genuinely* broken upload:
# we corrupt the signature in the presigned URL so MinIO refuses the PUT.
# (A dropped connection would look the same to the client: the PUT raises,
# the chunk row is never marked uploaded, and it stays 'pending'.)
print("\n📤 Simulating partial upload (chunks 0-2 succeed, 3-4 fail)...")
for info in upload2["presigned_urls"][:3]:  # only first 3
    idx = info["chunk_index"]
    _, chunk_data, _ = chunks[idx]
    etag = upload_chunk(info["url"], chunk_data)
    mark_chunk_uploaded(FILE_ID_2, idx, etag)
    print(f"   ✅ Chunk {idx} uploaded")

failed = []
for info in upload2["presigned_urls"][3:]:
    idx = info["chunk_index"]
    _, chunk_data, _ = chunks[idx]
    broken_url = info["url"].replace("Signature=", "Signature=CORRUPTED")
    try:
        upload_chunk(broken_url, chunk_data)
        print(f"   ⚠️  Chunk {idx} uploaded despite a bad signature (unexpected!)")
    except requests.HTTPError as e:
        failed.append(idx)
        print(f"   ❌ Chunk {idx} — upload rejected ({e.response.status_code}), still 'pending'")

# Check the status
print()
complete = finalize_upload(FILE_ID_2)  # should say incomplete

assert failed == [3, 4], f"chunks 3 and 4 were supposed to fail, failures were {failed}"
assert complete is False, "a file with pending chunks must NOT be finalized"

In [ ]:
# ── Resume the upload: find pending chunks and upload only those ──

def get_pending_chunks(file_id: int) -> list:
    """Query for chunks that haven't been uploaded yet."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("""
        SELECT chunk_index, storage_key
        FROM chunks
        WHERE file_id = %s AND status = 'pending'
        ORDER BY chunk_index
    """, (file_id,))
    pending = cur.fetchall()
    conn.close()
    return pending

# Find what still needs uploading
pending = get_pending_chunks(FILE_ID_2)
print(f"🔍 Found {len(pending)} pending chunks: {[p[0] for p in pending]}")
print()

# Resumability is the lesson: the server must remember exactly what is missing.
assert [p[0] for p in pending] == failed, (
    f"resume should target exactly the failed chunks {failed}, got {[p[0] for p in pending]}"
)

# Resume: upload only the pending chunks
s3 = get_s3_client()
print("📤 Resuming upload (only pending chunks)...")
for chunk_index, storage_key in pending:
    # Generate a fresh presigned URL
    url = s3.generate_presigned_url(
        ClientMethod="put_object",
        Params={"Bucket": BUCKET_NAME, "Key": storage_key},
        ExpiresIn=3600,
    )
    _, chunk_data, _ = chunks[chunk_index]
    etag = upload_chunk(url, chunk_data)
    mark_chunk_uploaded(FILE_ID_2, chunk_index, etag)
    print(f"   ✅ Chunk {chunk_index} uploaded (resumed)")

print()
complete = finalize_upload(FILE_ID_2)  # should succeed now!
print()
resent = len(pending)
print(f"💡 We only uploaded {resent} chunks instead of {len(chunks)} — "
      f"saved {(len(chunks) - resent) / len(chunks) * 100:.0f}% of the work!")

assert complete is True, "the file should finalize once every chunk is uploaded"

## 🧩 Assembling Chunks — Putting It All Back Together

Once all chunks are uploaded, we need to verify that the data is intact. The process is:

1. Read chunks from MinIO **in order** (chunk_0, chunk_1, chunk_2, ...)
2. Concatenate them back into the original file
3. Compute the SHA-256 of the reassembled file
4. Compare it against the fingerprint stored in Postgres

If the fingerprints match, we know **every byte** was transferred correctly.

In [ ]:
# ── Reassemble chunks from MinIO and verify integrity ──

def reassemble_file(file_id: int) -> bytes:
    """Download all chunks from MinIO in order and concatenate them."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("""
        SELECT chunk_index, storage_key FROM chunks
        WHERE file_id = %s ORDER BY chunk_index
    """, (file_id,))
    chunk_rows = cur.fetchall()
    conn.close()
    
    s3 = get_s3_client()
    assembled = b""
    for idx, key in chunk_rows:
        obj = s3.get_object(Bucket=BUCKET_NAME, Key=key)
        data = obj["Body"].read()
        assembled += data
        print(f"   📥 Read chunk {idx}: {len(data):,} bytes")
    
    return assembled

# Use the first file (FILE_ID) which had all chunks uploaded
print(f"🧩 Reassembling file {FILE_ID} from MinIO...")
reassembled = reassemble_file(FILE_ID)
print(f"\n   Total reassembled size: {len(reassembled):,} bytes")

In [ ]:
# ── Verify the fingerprint matches ──

reassembled_fp = hashlib.sha256(reassembled).hexdigest()

# Get the original fingerprint from Postgres
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT fingerprint FROM files WHERE id = %s", (FILE_ID,))
stored_fp = cur.fetchone()[0]
conn.close()

print("🔍 Fingerprint Verification")
print(f"   Original (in Postgres):  {stored_fp[:32]}...")
print(f"   Reassembled (computed):  {reassembled_fp[:32]}...")
print()

if reassembled_fp == stored_fp:
    print("✅ MATCH — The reassembled file is identical to the original!")
    print("   Every byte survived the chunk → upload → reassemble journey.")
else:
    print("❌ MISMATCH — Something went wrong during upload or reassembly.")

assert reassembled_fp == stored_fp, (
    f"reassembled file does not match the fingerprint Postgres recorded: "
    f"{reassembled_fp[:16]}... vs {stored_fp[:16]}..."
)
assert len(reassembled) == len(file_bytes), (
    f"reassembled {len(reassembled):,} bytes, original was {len(file_bytes):,}"
)

# Also confirm it matches our in-memory original
assert reassembled == file_bytes, "Reassembled bytes don't match original!"
print("✅ Byte-for-byte comparison with original data also passes!")

## 🧹 Cleanup

In [ ]:
# Clean up everything we created in this notebook

conn = get_db_connection()
cur = conn.cursor()

# Get all file IDs we created so we can delete their S3 objects
cur.execute("SELECT storage_key FROM chunks WHERE file_id IN (%s, %s)", (FILE_ID, FILE_ID_2))
keys_to_delete = [row[0] for row in cur.fetchall() if row[0]]

# Delete S3 objects
s3 = get_s3_client()
for key in keys_to_delete:
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=key)
    except Exception:
        pass
print(f"🧹 Deleted {len(keys_to_delete)} objects from MinIO")

# Delete DB records (chunks cascade from files)
cur.execute("DELETE FROM files WHERE id IN (%s, %s)", (FILE_ID, FILE_ID_2))
conn.commit()
conn.close()
print(f"🧹 Deleted file records {FILE_ID} and {FILE_ID_2} from Postgres")

# Clean up any Redis keys we might have set
r = get_redis_client()
for key in r.keys("upload:*"):
    r.delete(key)
print("🧹 Cleaned up Redis keys")
print()
print("✨ All clean! Ready for the next notebook.")

## 📚 Summary

### Key Takeaways

1. **Chunking is essential** — large files can't be uploaded in one request due to timeouts, payload limits, and network unreliability
2. **Split + fingerprint** — break the file into fixed-size chunks and compute a SHA-256 hash for each piece and the whole file
3. **Presigned URLs** — let clients upload directly to S3/MinIO without routing bytes through your backend (saves bandwidth and CPU)
4. **Database tracks state** — Postgres records which chunks are pending, uploading, or uploaded so the system always knows where things stand
5. **Resumable uploads** — if a transfer fails, query for pending chunks and only re-upload those (don't start from scratch!)
6. **Integrity verification** — reassemble chunks in order and compare the SHA-256 fingerprint to guarantee no data was lost or corrupted

### What's Happening Under the Hood

```
Client                        Backend API                  Postgres         MinIO (S3)
  │                               │                           │                │
  │  POST /upload/initiate        │                           │                │
  │ ─────────────────────────────>│  INSERT file + chunks     │                │
  │                               │ ─────────────────────────>│                │
  │  {file_id, presigned_urls[]}  │                           │                │
  │ <─────────────────────────────│                           │                │
  │                               │                           │                │
  │  PUT chunk_0 data ─────────────────────────────────────────────────────────>│
  │  PUT chunk_1 data ─────────────────────────────────────────────────────────>│
  │  ...                          │                           │                │
  │                               │                           │                │
  │  POST /upload/complete        │  UPDATE chunk statuses    │                │
  │ ─────────────────────────────>│ ─────────────────────────>│                │
  │                               │  UPDATE file status       │                │
  │  {status: "uploaded"} ✅      │ ─────────────────────────>│                │
  │ <─────────────────────────────│                           │                │
```

### Next Up

In **Notebook 2**, we'll implement **file sync & conflict resolution** — how changes made on one device propagate to others, and what happens when two devices edit the same file.